* https://drive.google.com/file/d/1todwi6NRInDwkFY24sSneP0c7RGTNTms/view?usp=sharing

#### konlpy 정상동작 확인

In [ ]:
!pip install konlpy

In [ ]:
!pip install JPype1

In [ ]:
import jpype
if not jpype.isJVMStarted():
    jpype.startJVM()
print("jpype와 JVM이 정상 작동 중임")

In [ ]:
from konlpy.tag import Okt

okt = Okt()
print(okt.morphs("KoNLPy가 정상 설치되었는지 확인 중입니다."))

In [ ]:
import jpype
if not jpype.isJVMStarted():
    jpype.startJVM()
print("JPype와 JVM이 정상 작동 중입니다.")


In [ ]:
import pandas as pd

#### 데이터 준비

In [ ]:
# ------------------------------
#   파일 경로
# ------------------------------
data_file = 'data/증권뉴스_2024-11-02.csv'

# ------------------------------
#   불러올 컬럼명
# ------------------------------
col_name = 'title'

# ------------------------------
#   데이터 불러오기
# ------------------------------
data = pd.read_csv(data_file).loc[:,col_name].dropna()

# ------------------------------
#   내용 컬럼을 하나의 문장으로 합치기
# ------------------------------
text = ' '.join(data)
text



#### 문자열 정제
* 한글만 남기기

In [ ]:
import re
clean_text = re.sub(r'[^가-힇\s]', ' ', text)
clean_text

* 블용어 사전

In [ ]:
# -----------------------
# 외부 불용어 사전
# -----------------------

import requests

url = "https://raw.githubusercontent.com/stopwords-iso/stopwords-ko/refs/heads/master/stopwords-ko.txt"
response = requests.get(url)

# 줄 단위로 나누고, 공백 제거 후 리스트 생성
external_stopwords = [line.strip() for line in response.text.splitlines() if line.strip()]

# -----------------------
# 커스터마이즈 불용어 사전
# -----------------------
custom_stopwords = []


# -----------------------
# 최종 불용어 사전
#   외부 불용어 사전, 커스터마이즈 불용어사전 합치고 중복 제거
# -----------------------
stopwords = set(external_stopwords + custom_stopwords)

#### 명사 추출 및 불용어 제거
* KoNLPy라이브러리의 형태소분석기 사용
* KoNLPy 사용 시 자바(JDK)가 필요할 수 있으니, 환경에 따라 JDK 설치 및 JAVA_HOME 환경변수 설정이 필요할 수도 있습니다​

In [ ]:
# ----------------------------
# 형태소 분석기로 명사만 추출
# ----------------------------

from konlpy.tag import Okt

okt = Okt()
nouns = okt.nouns(clean_text)

# ----------------------------
# 한글자인 명사와 불용어 제거
# ----------------------------

filtered_nouns = [
    n for n in nouns
    if len(n) > 1 and n not in stopwords
]

#### 단어 빈도 계산
* 리스트에서 각 항목이 몇 번 나오는지 카운트

In [ ]:
from collections import Counter

counter = Counter(filtered_nouns)  # 명사 리스트를 Counter 객체로 변환
print(counter.most_common(10))

#### 상위명사 추출

In [ ]:
top50 = dict(counter.most_common(50))  # 가장 많이 등장한 50개 단어만 추출하여 딕셔너리 생성

#### 한글 폰트 설정 및 워드클라우드 시각화

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

font_path = 'C:\\Windows\\Fonts\\Hancom Gothic Regular.ttf'  # 한글 폰트 경로 (환경에 맞게 변경)
wc = WordCloud(font_path=font_path, background_color='white', width=800, height=600)

# 앞서 계산한 단어 빈도 사용하여 워드클라우드 생성
cloud = wc.generate_from_frequencies(top50)  # 또는 wc.generate_from_frequencies(counter) 전체 단어 사용

# 워드클라우드 이미지 시각화
plt.figure(figsize=(10, 8))
plt.imshow(cloud, interpolation='bilinear')
plt.axis('off')  # 축 제거
plt.show()


In [ ]:

from matplotlib import font_manager

# 시스템에 설치된 모든 폰트 리스트 가져오기
font_list = font_manager.findSystemFonts(fontpaths=None, fontext='ttf')

# 한글 폰트만 필터링 (이름에 'Malgun', 'Gothic', 'Nanum', 'Apple' 등 포함)
korean_fonts = [f for f in font_list if 'malgun' in f.lower() or
                                         'gothic' in f.lower() or
                                         'nanum' in f.lower() or
                                         'apple' in f.lower()]

# 결과 확인
for font in korean_fonts:
    print(font)


In [ ]:
df = pd.read_csv(data_file)
df[df['title'].str.contains('비트코인')]['title']
